# Part 2: Predicting late deliveries at purchase time

**Question:** Can we flag, at the moment an order is placed, which orders are likely to arrive after the estimated delivery date?

**Why:** Part 1 showed that late orders drive a large share of 1 and 2 star reviews and cluster in Rio de Janeiro and the northeast. If risky orders can be flagged early, they could get proactive communication or a different carrier.

**Rules:** one row per delivered order; the target is `is_late` (delivered after the estimated date); only information available when the order is placed may be used as a feature; the train/test split is by time, not random.

## Feature rules (leakage control)

**Used (known when the order is placed):** customer and seller state, same-state flag, seller-to-customer distance, number of items, sellers and products, price, freight, weight, package volume, main category, payment type and instalments, purchase weekday and hour, and the promised delivery window.

**Excluded (not known yet):** approval and carrier handover dates, actual delivery date, delivery time, review score, and `purchase_month` (with under two years of data it would only memorise individual months).

**Split:** train on orders from Jan 2017 to Mar 2018, test on Apr to Aug 2018 (time-based, never random).

In [1]:
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

ROOT = Path.cwd().parent
DB = ROOT / "data" / "processed" / "olist.duckdb"
REPORTS = ROOT / "reports"

con = duckdb.connect(str(DB), read_only=True)
con.execute("SELECT COUNT(*) from delivered_orders").fetchone()

(96470,)

In [2]:
features = con.execute("""
    WITH geo AS (
        SELECT
            geolocation_zip_code_prefix AS zip,
            AVG(geolocation_lat) AS lat,
            AVG(geolocation_lng) AS lng
        FROM geolocation
        GROUP BY 1
    ),
    items AS (
        SELECT
            oi.order_id,
            COUNT(*)                      AS n_items,
            COUNT(DISTINCT oi.seller_id)  AS n_sellers,
            COUNT(DISTINCT oi.product_id) AS n_products,
            SUM(oi.price)                 AS total_price,
            SUM(oi.freight_value)         AS total_freight,
            SUM(p.product_weight_g)       AS total_weight_g,
            MAX(p.product_length_cm * p.product_height_cm * p.product_width_cm)
                                          AS max_volume_cm3,
            arg_max(COALESCE(t.product_category_name_english,
                             p.product_category_name, 'unknown'), oi.price)
                                          AS main_category,
            arg_max(s.seller_state, oi.price)           AS seller_state,
            arg_max(s.seller_zip_code_prefix, oi.price) AS seller_zip
        FROM order_items oi
        JOIN products p ON p.product_id = oi.product_id
        JOIN sellers s  ON s.seller_id = oi.seller_id
        LEFT JOIN product_category_name_translation t
               ON t.product_category_name = p.product_category_name
        GROUP BY 1
    ),
    pay AS (
        SELECT
            order_id,
            arg_max(payment_type, payment_value) AS main_payment_type,
            MAX(payment_installments)            AS max_installments
        FROM order_payments
        GROUP BY 1
    )
    SELECT
        d.order_id,
        d.order_purchase_timestamp                    AS purchase_ts,
        (d.days_vs_estimate > 0)::INT                 AS is_late,
        c.customer_state,
        i.seller_state,
        (c.customer_state = i.seller_state)::INT      AS same_state,
        i.n_items, i.n_sellers, i.n_products,
        i.total_price, i.total_freight,
        i.total_freight / NULLIF(i.total_price, 0)    AS freight_ratio,
        i.total_weight_g, i.max_volume_cm3,
        i.main_category,
        pay.main_payment_type, pay.max_installments,
        (epoch(d.order_estimated_delivery_date)
           - epoch(d.order_purchase_timestamp)) / 86400.0 AS est_days,
        month(d.order_purchase_timestamp)             AS purchase_month,
        dayofweek(d.order_purchase_timestamp)         AS purchase_dow,
        hour(d.order_purchase_timestamp)              AS purchase_hour,
        2 * 6371 * asin(sqrt(
            pow(sin(radians(gc.lat - gs.lat) / 2), 2)
            + cos(radians(gs.lat)) * cos(radians(gc.lat))
              * pow(sin(radians(gc.lng - gs.lng) / 2), 2)
        ))                                            AS distance_km
    FROM delivered_orders d
    JOIN customers c ON c.customer_id = d.customer_id
    JOIN items i     ON i.order_id = d.order_id
    LEFT JOIN pay    ON pay.order_id = d.order_id
    LEFT JOIN geo gc ON gc.zip = c.customer_zip_code_prefix
    LEFT JOIN geo gs ON gs.zip = i.seller_zip
""").df()

print(features.shape)
print("order_id unique:", features["order_id"].is_unique)
print("late rate:", round(100 * features["is_late"].mean(), 1), "%")
features.isna().mean().round(3).sort_values(ascending=False).head(8)

(96470, 22)
order_id unique: True
late rate: 6.8 %


distance_km       0.005
order_id          0.000
is_late           0.000
customer_state    0.000
seller_state      0.000
purchase_ts       0.000
same_state        0.000
n_items           0.000
dtype: float64

In [3]:
# Determine the exact missing counts
miss = features.isna().sum()
miss[miss > 0].sort_values(ascending=False).head(8)

distance_km          476
total_weight_g        16
max_volume_cm3        16
main_payment_type      1
max_installments       1
dtype: int64

In [4]:
# Determine monthly late delivery rates
monthly_late = (
    features.assign(month=features["purchase_ts"].dt.to_period("M"))
    .groupby("month")["is_late"]
    .agg(orders="size", late_rate=lambda s: round(100 * s.mean(), 1))
)
monthly_late

,orders,late_rate
month,,
2016-09,1,100.0
2016-10,265,0.8
2016-12,1,0.0
2017-01,750,2.9
2017-02,1653,3.0
2017-03,2546,4.6
2017-04,2303,6.6
2017-05,3545,3.0
2017-06,3135,3.0


In [5]:
# Confirm that the late rate is consistent across months
def late_rate_by(bucket):
    out = features.groupby(bucket, observed=True)["is_late"].agg(orders="size", late_rate="mean")
    out["late_rate"] = (100 * out["late_rate"]).round(1)
    return out

print("By seller-to-customer distance (km)")
display(late_rate_by(pd.cut(features["distance_km"], [0, 100, 500, 1000, 2000, 5000], include_lowest=True)))

print("By promised delivery window (days)")
display(late_rate_by(pd.cut(features["est_days"], [0, 10, 15, 20, 30, 60, 200])))

print("Seller and customer in the same state?")
display(late_rate_by(features["same_state"]))

By seller-to-customer distance (km)


,orders,late_rate
distance_km,,
"(-0.001, 100.0]",17897,4.4
"(100.0, 500.0]",36991,6.1
"(500.0, 1000.0]",25754,7.1
"(1000.0, 2000.0]",9739,9.5
"(2000.0, 5000.0]",5610,12.1


By promised delivery window (days)


,orders,late_rate
est_days,,
"(0, 10]",3945,10.5
"(10, 15]",10094,4.1
"(15, 20]",16625,7.1
"(20, 30]",45934,7.7
"(30, 60]",19636,5.0
"(60, 200]",236,2.5


Seller and customer in the same state?


,orders,late_rate
same_state,,
0,61780,8.0
1,34690,4.5


In [6]:
print((features["distance_km"] > 4500).sum(), "orders with an implausible distance (> 4,500 km)")
features.loc[features["distance_km"] > 4500, ["customer_state", "seller_state", "distance_km"]]

4 orders with an implausible distance (> 4,500 km)


,customer_state,seller_state,distance_km
23640,PA,SP,5338.619521
62192,BA,SP,4791.614270
74805,PA,SP,5338.619521
81703,PR,SC,8677.911622


In [7]:
period = features["purchase_ts"].dt.to_period("M").astype(str)
shock_months = ["2017-11", "2018-02", "2018-03"]   # months with late rate above ~10%
in_range = features["purchase_ts"] >= "2017-01-01"

groups = {
    "Calm months": features[in_range & ~period.isin(shock_months)],
    "Shock months (Nov 2017, Feb to Mar 2018)": features[period.isin(shock_months)],
}
for name, part in groups.items():
    print(f"{name}: {len(part):,} orders, late rate {100 * part['is_late'].mean():.1f}%")
    display(
        part.groupby(pd.cut(part["est_days"], [0, 10, 15, 20, 30, 60, 200]), observed=True)["is_late"]
        .agg(orders="size", late_rate=lambda s: round(100 * s.mean(), 1))
    )

Calm months: 75,357 orders, late rate 4.5%


,orders,late_rate
est_days,,
"(0, 10]",3734,10.6
"(10, 15]",7622,4.0
"(15, 20]",12419,4.1
"(20, 30]",35016,4.4
"(30, 60]",16403,3.8
"(60, 200]",163,3.1


Shock months (Nov 2017, Feb to Mar 2018): 20,846 orders, late rate 15.1%


,orders,late_rate
est_days,,
"(0, 10]",211,8.5
"(10, 15]",2472,4.6
"(15, 20]",4205,16.0
"(20, 30]",10916,18.4
"(30, 60]",3020,11.5
"(60, 200]",22,4.5


In [8]:
# Split into train and test sets - January 2017 to March 2018 for training, April 2018 onwards for testing
model_df = features[features["purchase_ts"] >= "2017-01-01"].copy()
model_df.loc[model_df["distance_km"] > 4500, "distance_km"] = np.nan

CUTOFF = pd.Timestamp("2018-04-01")
train = model_df[model_df["purchase_ts"] < CUTOFF].copy()
test = model_df[model_df["purchase_ts"] >= CUTOFF].copy()

for name, part in [("train", train), ("test", test)]:
    print(f"{name}: {len(part):,} orders, {part['is_late'].sum():,} late "
          f"({100 * part['is_late'].mean():.1f}%), "
          f"{part['purchase_ts'].min().date()} to {part['purchase_ts'].max().date()}")

NUMERIC = ["n_items", "n_sellers", "n_products", "total_price", "total_freight",
           "freight_ratio", "total_weight_g", "max_volume_cm3", "max_installments",
           "est_days", "purchase_dow", "purchase_hour", "distance_km", "same_state"]
CATEGORICAL = ["customer_state", "seller_state", "main_category", "main_payment_type"]
TARGET = "is_late"

base_rate_test = test[TARGET].mean()
print(f"\nAlways-predict-on-time accuracy on test: {100 * (1 - base_rate_test):.1f}%")
print(f"PR-AUC of a model with no skill (= base rate): {base_rate_test:.3f}")

train: 64,053 orders, 5,110 late (8.0%), 2017-01-05 to 2018-03-31
test: 32,150 orders, 1,421 late (4.4%), 2018-04-01 to 2018-08-29

Always-predict-on-time accuracy on test: 95.6%
PR-AUC of a model with no skill (= base rate): 0.044


In [9]:
# Categorical gaps become an explicit "missing" category
for part in (train, test):
    part[CATEGORICAL] = part[CATEGORICAL].fillna("missing")

LOG_COLS = ["total_price", "total_freight", "freight_ratio", "total_weight_g", "max_volume_cm3"]
PLAIN_COLS = [c for c in NUMERIC if c not in LOG_COLS]


def make_preprocessor():
    log_pipe = Pipeline([
        ("log", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
        ("impute", SimpleImputer(strategy="median", add_indicator=True)),
        ("scale", StandardScaler()),
    ])
    plain_pipe = Pipeline([
        ("impute", SimpleImputer(strategy="median", add_indicator=True)),
        ("scale", StandardScaler()),
    ])
    cat_pipe = OneHotEncoder(handle_unknown="ignore", min_frequency=200, sparse_output=False)
    return ColumnTransformer([
        ("log", log_pipe, LOG_COLS),
        ("plain", plain_pipe, PLAIN_COLS),
        ("cat", cat_pipe, CATEGORICAL),
    ])


def evaluate(name, y_true, scores, train_auc, top_frac=0.10):
    y = np.asarray(y_true)
    n_top = int(top_frac * len(y))
    top = np.argsort(-scores)[:n_top]
    return {
        "model": name,
        "train ROC-AUC": round(train_auc, 3),
        "test ROC-AUC": round(roc_auc_score(y, scores), 3),
        "test PR-AUC": round(average_precision_score(y, scores), 3),
        "late orders caught in top 10% of risk": f"{100 * y[top].sum() / y.sum():.1f}%",
        "lift in top 10%": round(y[top].mean() / y.mean(), 2),
    }


models = {
    "Baseline (constant prediction)": DummyClassifier(strategy="prior"),
    "Logistic regression": LogisticRegression(max_iter=2000),
    "Gradient boosting": HistGradientBoostingClassifier(
        learning_rate=0.05, max_iter=200, max_leaf_nodes=15,
        min_samples_leaf=100, l2_regularization=1.0, random_state=42,
    ),
}

X_train, y_train = train[NUMERIC + CATEGORICAL], train[TARGET]
X_test, y_test = test[NUMERIC + CATEGORICAL], test[TARGET]

results, fitted = [], {}
for name, clf in models.items():
    pipe = Pipeline([("prep", make_preprocessor()), ("model", clf)])
    pipe.fit(X_train, y_train)
    train_auc = roc_auc_score(y_train, pipe.predict_proba(X_train)[:, 1])
    scores = pipe.predict_proba(X_test)[:, 1]
    fitted[name] = pipe
    results.append(evaluate(name, y_test, scores, train_auc))

pd.DataFrame(results)

,model,train ROC-AUC,test ROC-AUC,test PR-AUC,late orders caught in top 10% of risk,lift in top 10%
0,Baseline (constant prediction),0.500,0.500,0.044,10.1%,1.01
1,Logistic regression,0.716,0.704,0.091,24.1%,2.41
2,Gradient boosting,0.776,0.673,0.083,22.3%,2.23


In [10]:
rng = np.random.default_rng(42)
scores_lr = fitted["Logistic regression"].predict_proba(X_test)[:, 1]
scores_gb = fitted["Gradient boosting"].predict_proba(X_test)[:, 1]
y = y_test.to_numpy()

diffs, aucs_lr = [], []
for _ in range(300):
    idx = rng.integers(0, len(y), len(y))
    if y[idx].sum() == 0:
        continue
    a_lr = roc_auc_score(y[idx], scores_lr[idx])
    a_gb = roc_auc_score(y[idx], scores_gb[idx])
    aucs_lr.append(a_lr)
    diffs.append(a_lr - a_gb)

lo, hi = np.percentile(aucs_lr, [2.5, 97.5])
print(f"Logistic regression test ROC-AUC: 95% interval {lo:.3f} to {hi:.3f}")
lo, hi = np.percentile(diffs, [2.5, 97.5])
print(f"LR minus GB ROC-AUC difference: mean {np.mean(diffs):.3f}, 95% interval {lo:.3f} to {hi:.3f}")

Logistic regression test ROC-AUC: 95% interval 0.691 to 0.716
LR minus GB ROC-AUC difference: mean 0.030, 95% interval 0.021 to 0.039


In [11]:
from sklearn.base import clone

SHOCK = ["2017-11", "2018-02", "2018-03"]
train_period = train["purchase_ts"].dt.to_period("M").astype(str)
train_calm = train[~train_period.isin(SHOCK)]
print(f"calm-only training set: {len(train_calm):,} orders, late rate {100 * train_calm[TARGET].mean():.1f}%")

rows = []
for label, part in [("all training months", train), ("calm months only", train_calm)]:
    for name in ["Logistic regression", "Gradient boosting"]:
        pipe = Pipeline([("prep", make_preprocessor()), ("model", clone(models[name]))])
        pipe.fit(part[NUMERIC + CATEGORICAL], part[TARGET])
        tr_auc = roc_auc_score(part[TARGET], pipe.predict_proba(part[NUMERIC + CATEGORICAL])[:, 1])
        scores = pipe.predict_proba(X_test)[:, 1]
        row = evaluate(name, y_test, scores, tr_auc)
        row["trained on"] = label
        rows.append(row)

pd.DataFrame(rows)[["trained on", "model", "train ROC-AUC", "test ROC-AUC", "test PR-AUC",
                    "late orders caught in top 10% of risk", "lift in top 10%"]]

calm-only training set: 43,207 orders, late rate 4.5%


,trained on,model,train ROC-AUC,test ROC-AUC,test PR-AUC,late orders caught in top 10% of risk,lift in top 10%
0,all training months,Logistic regression,0.716,0.704,0.091,24.1%,2.41
1,all training months,Gradient boosting,0.776,0.673,0.083,22.3%,2.23
2,calm months only,Logistic regression,0.684,0.674,0.083,23.2%,2.32
3,calm months only,Gradient boosting,0.740,0.688,0.087,24.1%,2.41


## Development results

| Trained on | Model | Test ROC-AUC | Test PR-AUC | Late orders in top 10% of risk |
|---|---|---|---|---|
| All months | Logistic regression | 0.704 | 0.091 | 24.1% |
| All months | Gradient boosting | 0.673 | 0.083 | 22.3% |
| Calm months only | Logistic regression | 0.674 | 0.083 | 23.2% |
| Calm months only | Gradient boosting | 0.688 | 0.087 | 24.1% |

- Logistic regression beats the first gradient boosting model: ROC-AUC difference 0.030 (95% bootstrap interval 0.021 to 0.039).
- Gradient boosting overfits (train 0.776 vs test 0.673). Training on calm months only halves that gap, so part of the problem is distribution shift, but the same change costs logistic regression 0.030 because it loses about 60% of its late-order examples.
- Dropping shock months is therefore not a fix. Next: choose settings with time-ordered validation inside the training period, then score the chosen model on the test set once.